# Level 3 — Functional Agent / Lead Generation & Targeted Marketing

This notebook demonstrates the Level 3 Functional Agent capabilities:

- **UC1**: Lead Scoring — identify top prospects for a product offer
- **UC2**: Customer Enrichment — multi-source data (credit bureau, business registry, location)
- **UC3**: Next-Best-Action (NBA) Recommendation — ranked offer scoring
- **UC4**: Consent-Gated Notification — send offer email with consent check
- **UC5**: Identity Gate — block action on unverified identity, open remediation case
- **UC6**: Bulk Campaign Targeting — segment-level execution plan
- **UC7**: Return Risk Intervention — identify and contact high-return-risk customers
- **UC8**: Campaign Results Dashboard — analyse past campaign performance
- **UC9**: Guardrails — PII redaction and policy enforcement

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from notebooks.utils import (
    display_card, display_metrics, display_bar_chart,
    display_side_by_side, display_violations_list, display_status_banner,
    display_lead_scoring_results, display_customer_enrichment,
    display_nba_recommendations, display_consent_notification, display_blocked_example,
    display_kyc_gate, display_campaign_execution_plan, display_return_risk_intervention,
    display_campaign_results_dashboard, display_title
)
from IPython.display import display, HTML
import json, pandas as pd
from pathlib import Path

from src.tools.leads import score_leads, enrich_customer, bulk_recommend
from src.tools.functional import recommend_offer, draft_email, send_notification, create_case
from src.tools.customer import search_customer_profile, get_kyc_status

print("✅ Level 3 Functional Agent — ready")


✅ Level 3 Functional Agent — ready


## UC1: Lead Scoring — Identify Top Prospects

In [2]:
# UC1: Lead Scoring — identify top prospects for Premium Membership
offer_code = "PROMO-PREMIUM-MEMBERSHIP"
result = score_leads(offer_code, top_n=10)

display_lead_scoring_results(result)


Customer ID,Name,Segment,Lead Score,Rationale
CUST-034,Customer CUST-034,vip,0.911,high engagement (84%); premium segment (vip)
CUST-173,Customer CUST-173,vip,0.895,high engagement (85%); premium segment (vip)
CUST-030,Customer CUST-030,vip,0.880,high engagement (80%); premium segment (vip)
CUST-176,Customer CUST-176,vip,0.869,high engagement (79%); premium segment (vip)
CUST-100,Customer CUST-100,vip,0.864,high engagement (76%); premium segment (vip)
CUST-060,Customer CUST-060,vip,0.863,high engagement (78%); premium segment (vip)
CUST-146,Customer CUST-146,vip,0.858,electronics buyer — premium membership fits; high
CUST-107,Customer CUST-107,vip,0.852,high engagement (78%); premium segment (vip)
CUST-118,Customer CUST-118,vip,0.852,high engagement (85%); premium segment (vip)
CUST-047,Customer CUST-047,vip,0.848,premium segment (vip)


## UC2: Customer Enrichment — Multi-Source Data

In [3]:
# UC2: Customer Enrichment — multi-source data enrichment
customer_id = "CUST-021"
enriched = enrich_customer(customer_id)

display_customer_enrichment(enriched, customer_id)


## UC3: Next-Best-Action (NBA) Recommendation

In [4]:
# UC3: Next-Best-Action (NBA) Recommendation — ranked offer scoring
test_customers = ["CUST-021", "CUST-017", "CUST-041", "CUST-079", "CUST-143"]

display_nba_recommendations(test_customers, recommend_offer, search_customer_profile)


Customer ID,Segment,Best Offer,Confidence
CUST-021,vip,Bundle Deal,0.879
CUST-017,at_risk,Premium Membership,0.457
CUST-041,dormant_vip,Premium Membership,0.749
CUST-079,casual,Win-Back Offer,0.834
CUST-143,casual,Win-Back Offer,0.612


## UC4: Consent-Gated Notification

In [5]:
# UC4: Consent-Gated Notification — send offer to eligible customer
customer_id = "CUST-021"  # vip, marketing consent=true

offer = recommend_offer(customer_id)
draft = draft_email(customer_id, "T-PROMO-01", variables={"offer_name": offer["offer_name"]})
send_result = send_notification(customer_id, "email", {"subject": draft["subject"], "body": draft["body"]})

display_consent_notification(customer_id, send_result, draft)

# Blocked example — CUST-072 has marketing consent=false
blocked_id = "CUST-072"
blocked_result = send_notification(blocked_id, "email", {"subject": "Test", "body": "Test"})
display_blocked_example(blocked_id, blocked_result)


## UC5: Identity Gate — Block Action on Unverified Identity

In [6]:
# UC5: Identity Gate — block action on unverified identity, open remediation case
customer_id = "CUST-013"  # identity_status=unverified

kyc = get_kyc_status(customer_id)

case = None
if kyc.get("kyc_status") in ("unverified", "expired"):
    case = create_case(customer_id, "identity_reverification",
                       "Identity unverified — re-verification required before action", priority="high")

display_kyc_gate(customer_id, kyc, case)


## UC6: Bulk Campaign Targeting

In [7]:
# UC6: Bulk Campaign Targeting — segment-level offer execution plan
offer_code = "PROMO-WINBACK"
segment = "dormant_vip"

plan = bulk_recommend(offer_code, segment=segment, top_n=50)

display_campaign_execution_plan(plan, offer_code, segment)


## UC7: Return Risk Intervention

In [8]:
# UC7: Return Risk Intervention — identify high-return-risk customers and send win-back offers
customers = json.loads(Path("../data/customers.json").read_text())

at_risk = [
    c for c in customers
    if c.get("return_risk", 0) > 0.7
    and c.get("identity_status") in ("verified", "pending")
    and c.get("consent_flags", {}).get("marketing", False)
]
at_risk.sort(key=lambda x: x["return_risk"], reverse=True)

display_return_risk_intervention(at_risk, send_notification)


Customer ID,Segment,Return Risk,Status
CUST-014,at_risk,0.83,blocked
CUST-089,dormant_vip,0.80,sent
CUST-052,dormant_vip,0.79,sent
CUST-037,dormant_vip,0.78,blocked
CUST-069,dormant_vip,0.76,sent
CUST-132,dormant_vip,0.75,sent
CUST-185,at_risk,0.75,sent
CUST-123,at_risk,0.74,sent


## UC8: Campaign Results Dashboard

In [9]:
# UC8: Campaign Results Dashboard — analyse past campaign performance
campaign_results = json.loads(Path("../data/campaign_results.json").read_text())

display_campaign_results_dashboard(campaign_results)


## UC9: Guardrails — PII Redaction & Policy Enforcement

In [11]:
# UC9: Guardrails — PII Redaction & Policy Enforcement

import os
from src.core.guardrails import guardrail_check

os.environ["GUARDRAIL_ENABLED"] = "true"

test_output = "Customer email is john.doe@example.com and phone is 555-123-4567. We recommend buying shares."

result = guardrail_check(test_output, request_id="demo-001")

display_title("Guardrail Check", emoji="🛡️")

display_side_by_side(
    "❌ Original",
    test_output,
    "✅ Sanitized",
    result.revised_text,
    left_color="#fee2e2",
    left_text_color="#dc2626",
    right_color="#dcfce7",
    right_text_color="#16a34a"
)

display_violations_list(result.violations)

display_metrics({
    "Guardrail Passed": "✅ Yes" if result.passed else "❌ No",
    "Violations Found": len(result.violations),
    "Action Taken": "Redacted PII & forbidden phrases",
})

os.environ["GUARDRAIL_ENABLED"] = "false"
display_status_banner("✅ Guardrails disabled for remaining demos", status="success")
